# Task 1 — Exploratory analysis (FNSPID-style news)

This notebook profiles **headline text**, **publishers**, and **publication timing** for the financial news file in `data/raw/`. Replace `fnspid_sample.csv` with your full FNSPID extract; column names should match the assignment (`headline`, `url`, `publisher`, `date`, `stock`).


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 4)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

NEWS_PATH = ROOT / "data" / "raw" / "fnspid_sample.csv"
news = pd.read_csv(NEWS_PATH)
news.head()


## Descriptive statistics — headline length


In [ ]:
news["headline_len"] = news["headline"].astype(str).str.len()
news["headline_words"] = news["headline"].astype(str).str.split().str.len()

display(news[["headline_len", "headline_words"]].describe())

fig, ax = plt.subplots()
sns.histplot(news["headline_len"], bins=30, kde=True, ax=ax)
ax.set_title("Distribution of headline character counts")
ax.set_xlabel("Characters")
plt.tight_layout()
plt.show()


## Publisher activity


In [ ]:
pub_counts = news["publisher"].astype(str).value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=pub_counts.values, y=pub_counts.index, ax=ax, orient="h", palette="viridis")
ax.set_title("Top publishers by article count")
ax.set_xlabel("Articles")
plt.tight_layout()
plt.show()


## Publication calendar — volume and intraday timing


In [ ]:
# Strip timezone label for parsing; real FNSPID rows include 'UTC-4' text.
clean_dates = (
    news["date"]
    .astype(str)
    .str.replace(r"\s*UTC-4\s*$", "", regex=True)
)
news["pub_ts"] = pd.to_datetime(clean_dates, errors="coerce")
missing_ts = news["pub_ts"].isna().mean()
print(f"Share of rows with unparseable timestamps: {missing_ts:.2%}")

news["pub_date"] = news["pub_ts"].dt.normalize()
news["pub_hour"] = news["pub_ts"].dt.hour

daily = (
    news.dropna(subset=["pub_date"])
    .groupby("pub_date")
    .size()
    .sort_index()
)
monthly = daily.resample("ME").sum()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
monthly.plot(ax=axes[0], color="steelblue", title="Monthly article count (sample data)")
axes[0].set_ylabel("Articles")
axes[0].set_xlabel("")

news["pub_hour"].dropna().astype(int).plot(kind="hist", bins=24, ax=axes[1], color="darkorange", edgecolor="white")
axes[1].set_title("Hour of day (parsed local clock) — when do headlines land?")
axes[1].set_xlabel("Hour")
plt.tight_layout()
plt.show()


### Relating spikes to market events

On the full dataset, overlay known catalysts (FOMC, CPI, mega-cap earnings clusters) by annotating peaks in the monthly series. The sample generator does not encode real macro events — treat this as a template.


## Keywords & topics (TF-IDF + LDA)


In [ ]:
headlines = news["headline"].astype(str).fillna("")

vectorizer = TfidfVectorizer(
    max_features=40,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
)
try:
    X = vectorizer.fit_transform(headlines)
    scores = np.asarray(X.mean(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    top_idx = scores.argsort()[::-1][:20]
    display(pd.DataFrame({"term": terms[top_idx], "mean_tfidf": scores[top_idx]}))
except ValueError as e:
    print("TF-IDF skipped (too few documents for min_df):", e)

# LDA on word counts (unsupervised themes)
cv = CountVectorizer(max_features=200, stop_words="english", min_df=2)
try:
    Xc = cv.fit_transform(headlines)
    lda = LatentDirichletAllocation(n_components=4, random_state=42, max_iter=20, learning_method="batch")
    lda.fit(Xc)
    vocab = cv.get_feature_names_out()
    for i, topic in enumerate(lda.components_):
        top = topic.argsort()[::-1][:12]
        print(f"Topic {i+1}: ", ", ".join(vocab[j] for j in top))
except ValueError as e:
    print("LDA skipped:", e)


## Publisher domains (email-style identifiers)


In [ ]:
import re

pub = news["publisher"].astype(str)

def domain(name: str) -> str | None:
    m = re.search(r"@([\w.-]+)", name)
    return m.group(1).lower() if m else None

news["pub_domain"] = pub.map(domain)
domain_counts = news["pub_domain"].dropna().value_counts().head(12)
display(domain_counts.to_frame("articles"))


## Takeaways for downstream modeling

- **Text**: headline length and vocabulary prime the choice of tokenizer and whether bigrams are needed for phrases like *price target*.
- **Time**: intraday spikes inform whether same-day returns should align to the **next** session open vs close.
- **Publishers**: dominant feeds may introduce **source bias**; consider de-duplication or publisher fixed effects before correlation work.

Next: merge with prices (`02_task2_technical_indicators.ipynb`) and quantify sentiment vs returns (`03_task3_sentiment_correlation.ipynb`).
